# lesson09 homework｜Selenium Notebook

先從上到下執行設定和函式定義格。每次示範或檢查會啟動練習站及 Chrome，結束時會關閉；請等待執行完成後再執行下一格。


In [ ]:

import os
import sys
import time
from pathlib import Path

import os
from pathlib import Path

# 從教材根目錄或本堂資料夾啟動 Notebook 都可以。
current = Path.cwd().resolve()
lesson_name = 'lesson09_selenium_basics'
candidates = [current, *current.parents]
lesson_path = next((p for p in candidates if p.name == lesson_name and (p / 'fixtures').is_dir()), None)
if lesson_path is None:
    lesson_path = next((p / 'lessons' / lesson_name for p in candidates if (p / 'lessons' / lesson_name / 'fixtures').is_dir()), None)
if lesson_path is None:
    raise FileNotFoundError('找不到本堂教材。請用 VS Code 開啟完整教材資料夾，重新啟動核心後再執行。')
os.chdir(lesson_path)
print('工作資料夾：', Path.cwd())
sys.path.insert(0, str(Path.cwd().parents[1] / "tools"))

from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.support.ui import WebDriverWait

from practice_site import TEST_PASSWORD, TEST_USER, start_server, stop_server

HEADLESS = os.environ.get("HEADLESS") == "1"




## new_driver


In [ ]:
def new_driver():
    options = webdriver.ChromeOptions()
    if HEADLESS:
        options.add_argument("--headless=new")
    options.add_argument("--window-size=1280,900")
    return webdriver.Chrome(options=options)


## do_login


In [ ]:
def do_login(driver, base_url):
    """已經寫好的登入函式，H2 可以直接使用。"""
    driver.get(f"{base_url}/login")
    driver.find_element(By.ID, "username").send_keys(TEST_USER)
    driver.find_element(By.ID, "password").send_keys(TEST_PASSWORD)
    driver.find_element(By.ID, "login-button").click()
    WebDriverWait(driver, 10).until(EC.presence_of_element_located((By.ID, "welcome")))


## h1_search_titles


In [ ]:
def h1_search_titles(driver, base_url, keyword):
    """打開搜尋頁，搜尋 keyword（輸入後點擊搜尋按鈕），等到網址包含 "q="（新頁面已載入）。
    回傳結果裡每一筆課程名稱（class 為 title 的 span 的文字），list，依頁面順序。
    提示：網址裡的中文會被編碼（例如 q=%E5%85%A5%E9%96%80），所以不能等 "q=入門"，只等 "q=" 就好。"""
    # TODO: 請在這裡寫程式
    return None


## h2_unpaid_total


In [ ]:
def h2_unpaid_total(driver, base_url):
    """用 do_login 登入，讀取會員頁訂單表（id 為 orders）的每一列，
    把狀態是「待付款」的金額加總，回傳整數。
    提示：每一列有 4 個 td：訂單、課程、金額、狀態。"""
    # TODO: 請在這裡寫程式
    return None


## h3_timed_wait


In [ ]:
def h3_timed_wait(driver, base_url):
    """打開 base_url + "/delayed"，用明確等待等到 id 為 result 的元素出現。
    回傳 (元素文字, 從打開網頁到元素出現花了幾秒) 這個 tuple。"""
    # TODO: 請在這裡寫程式
    return None


## check_h3


In [ ]:
def check_h3(result):
    text, seconds = result
    return text == "資料載入完成：共 3 筆" and 1.5 <= seconds < 8


## main


In [ ]:
def main():
    checks = [
        ("H1", "搜尋「入門」", lambda d, b: h1_search_titles(d, b, "入門") == ["Python 入門課", "SQL 入門", "機器學習入門"]),
        ("H2", "待付款金額", lambda d, b: h2_unpaid_total(d, b) == 3900),
        ("H3", "量測等待時間", lambda d, b: check_h3(h3_timed_wait(d, b))),
    ]

    server, base_url = start_server()
    driver = None
    failed = 0
    try:
        driver = new_driver()
        for number, description, check in checks:
            driver.get("about:blank")
            driver.delete_all_cookies()
            started = time.time()
            try:
                passed = bool(check(driver, base_url))
                note = ""
            except Exception as error:  # noqa: BLE001 - show every kind of mistake
                passed = False
                note = f"{type(error).__name__}: {str(error).splitlines()[0] if str(error) else ''}"
            failed += 0 if passed else 1
            print(f"{number} {'PASS' if passed else 'FAIL'}  {description}  ({time.time() - started:.1f}s) {note}")
    finally:
        if driver is not None:
            driver.quit()
        stop_server(server)

    print(f"\n通過 {len(checks) - failed} / {len(checks)} 題")
    sys.exit(1 if failed else 0)


## 執行檢查
完成上方 TODO 並執行相關函式格後，再執行本格。


In [ ]:
try:
    main()
except SystemExit as result:
    print("檢查結束；狀態：", result.code)
